# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [ ]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_" + datetime_str + ".zarr"

# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_014357.zarr"
# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_111309.zarr"
# output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_133357.zarr'
# output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_165714.zarr'
output_file = './results/patrolling_comms_test5.zarr'
# output_file = './results/patrolling_comms_test5_commsAdjusted.zarr'

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 25

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(1)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(3)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [ ]:
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = [50.0, 100.0, 150.0, 200.0]
comms_probability = 0.048

import numpy as np

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    for obs_radius in obs_radii:
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-draft1/wandb/run-20260428_120003-qjlprpbg/files", default_args)
        args["eval_series"] = "MAGEC"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-pred-draft1/wandb/run-20260428_115953-i37v2u1j/files", default_args)
        args["eval_series"] = "Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-pred-uq-draft1/wandb/run-20260428_115938-2o5oq904/files", default_args)
        args["eval_series"] = "Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-comms-pred-draft1/wandb/run-20260428_115929-oucs69i8/files", default_args)
        args["eval_series"] = "Learned Comms. + Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-comms-pred-uq-draft1/wandb/run-20260427_193746-erwygzow/files", default_args)
        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-comms-pred-uq-draft1/wandb/run-20260428_011915-6z90l1c2/files", default_args)
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-comms-pred-uq-draft1/wandb/run-20260428_115918-5m7c4obj/files", default_args)
        args["eval_series"] = f"Learned Comms. + Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/diffuser-patrolling-random-obs200-uqEnsemble3-pygNS-commsRL1.0-injectionAppend-predObs/wandb/run-20260426_231156-ghiad0kt/files", default_args)
        # args["eval_series"] = f"Learned Comms. + Prediction + UQ ({args['experiment_name']['value']})"
        # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        # args["observation_radius"] = wrap_arg_val(obs_radius)
        # args["communication_probability"] = wrap_arg_val(comms_probability)
        # start_process(args)
        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/diffuser-patrolling-random-obs200-uqEnsemble3-pygNS-commsRL0.1-injectionAppend-predObs/wandb/run-20260426_231156-kqzdpjb5/files", default_args)
        # args["eval_series"] = f"Learned Comms. + Prediction + UQ ({args['experiment_name']['value']})"
        # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        # args["observation_radius"] = wrap_arg_val(obs_radius)
        # args["communication_probability"] = wrap_arg_val(comms_probability)
        # start_process(args)
        # # Autoregression fix
        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-comms-pred-uq-draft1/wandb/run-20260427_013623-oo6ntxc2/files", default_args)
        # args["eval_series"] = f"Learned Comms. + Prediction + UQ ({args['experiment_name']['value']})"
        # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        # args["observation_radius"] = wrap_arg_val(obs_radius)
        # args["communication_probability"] = wrap_arg_val(comms_probability)
        # start_process(args)


    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

In [ ]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_radius'

# Data for plotting
plots = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'], axis=tuple(range(1, output[series][exp]['avg_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'][:, -1], axis=tuple(range(1, output[series][exp]['avg_idleness'][:, -1].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['avg_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        # 'y_std': {
        #     series: {
        #         exp: np.std(output[series][exp]['avg_idleness'], axis=0)
        #         for exp in sorted_experiments(series, output, x_key)
        #     }
        #     for series in output
        # }
    },
    'worst_idleness': {
        'title': 'Time-Averaged Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['worst_idleness'], axis=tuple(range(1, output[series][exp]['worst_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_final': {
        'title': 'Final Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_over_time': {
        'title': 'Worst Idleness vs. Time',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'stddev_idleness': {
        'title': 'Time-Averaged Std. Dev. Idleness',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['stddev_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'stddev_idleness_over_time': {
        'title': 'Std. Dev. Idleness vs. Time',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'communication_requests': {
        'title': 'Communication Requests vs. Observation Radius',
        'ylabel': 'Communication Requests',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['communication']['requests_made']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'communication_requests_time': {
        'title': 'Cumulative Communication Requests vs. Time',
        'ylabel': 'Communication Requests',
        'xlabel': 'Time Steps',
        'y': {
            series: {
                exp: np.cumsum(np.mean(output[series][exp]['communication']['requests_made'], axis=0))
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

plot(plots, timesteps=timesteps)
